# [Chapter 8 — Parameter Estimation, §8.5] Fitting $F_0$: The Initial Susceptible Fraction

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 8 — Parameter Estimation
**Considerations developed:** 8, 9 (practical fitting)
**Estimated runtime:** ≤ 1 minute

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/)

## What this notebook does
What if we don't assume $S^*$ but fit it as a free parameter? Demonstrates that joint fitting of $\mathcal{R}_0$ and $F_0 = S_0/N$ is often practically non-identifiable from incidence data alone.

## What you should already know
Chapter 8 central comparison notebook.


## Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import matplotlib.pyplot as plt
from shared import (
    book_style,
    BOOK_COLORS,
    integrate_sir_i,
    baseline_central_comparison,
)
from shared.parameters import baseline_chapter_08
from shared.seeds import set_seed_chapter_08
from shared.verification import assert_within_tolerance

set_seed_chapter_08()
book_style()


## The joint fit

Instead of assuming $S^*$, we treat $F_0 = S_0/N$ as a free parameter and fit both $\mathcal{R}_0$ and $F_0$ simultaneously by minimizing residuals.

The book's caveat: this is often a *practically non-identifiable* problem — many $(\mathcal{R}_0, F_0)$ pairs produce nearly identical incidence curves. The joint fit can succeed numerically but produce wide confidence regions.

In [ ]:
from scipy.optimize import minimize

params = baseline_central_comparison()
true_R0 = params['true_R_0']
true_F0 = params['S0']

# Generate observed data
result = integrate_sir_i(params)
t = result['t']
S, I = result['S'], result['I']
alpha_true = params['c_I'] * params['beta'] * (S / params['N'])
J_obs = np.maximum(alpha_true * I + params['noise_sigma'] * (alpha_true * I).max() * np.random.randn(len(t)), 0)

# Fit: vary R_0 and F_0, hold tau_R fixed
def simulate(R0_val, F0_val):
    p = params.copy()
    p['c_I'] = R0_val / (params['beta'] * params['tau_R'])
    p['S0'] = F0_val
    p['I0'] = 0.001
    p['R0_init'] = 1.0 - F0_val - p['I0']
    res = integrate_sir_i(p)
    alpha_t = p['c_I'] * p['beta'] * (res['S'] / p['N'])
    return alpha_t * res['I']

def loss(theta):
    R0_val, F0_val = theta
    if R0_val <= 0.5 or F0_val <= 0.1 or F0_val >= 1.0:
        return 1e10
    J_pred = simulate(R0_val, F0_val)
    return np.sum((J_pred - J_obs)**2)

# Try fitting from a few initial guesses
results = []
for init in [(1.5, 0.8), (2.5, 0.95), (3.0, 0.6)]:
    res = minimize(loss, init, method='Nelder-Mead', options={'xatol':1e-3, 'fatol':1e-8})
    results.append(res.x)
    print(f'From init {init}: fitted R_0 = {res.x[0]:.3f}, F_0 = {res.x[1]:.3f}, loss = {res.fun:.4e}')

print(f'\nTrue values: R_0 = {true_R0:.3f}, F_0 = {true_F0:.3f}')


## Identifiability

Visualize the loss surface in the $(\mathcal{R}_0, F_0)$ plane to see how identifiable (or not) the joint fit is.

In [ ]:
# Compute loss on a grid
R0_grid = np.linspace(1.5, 2.6, 22)
F0_grid = np.linspace(0.6, 1.0, 21)
R0_mesh, F0_mesh = np.meshgrid(R0_grid, F0_grid)
loss_grid = np.zeros_like(R0_mesh)
for i in range(R0_mesh.shape[0]):
    for j in range(R0_mesh.shape[1]):
        loss_grid[i,j] = loss((R0_mesh[i,j], F0_mesh[i,j]))

fig, ax = plt.subplots(figsize=(7, 5))
log_loss = np.log10(loss_grid - loss_grid.min() + 1e-12)
im = ax.contourf(R0_mesh, F0_mesh, log_loss, levels=20, cmap='viridis')
plt.colorbar(im, ax=ax, label=r'$\log_{10}$(loss - min)')
ax.plot(true_R0, true_F0, '*', color='white', ms=15, mec='black', label='True')
for r in results:
    ax.plot(r[0], r[1], 'o', color=BOOK_COLORS['lambda_hat'], ms=8, mec='black')
ax.set_xlabel(r'$R_0$')
ax.set_ylabel(r'$F_0 = S_0/N$')
ax.set_title('Loss surface — note the elongated valley (practical non-identifiability)')
ax.legend()
plt.tight_layout()
plt.show()


## What's next

The next notebook applies this comparison to a realistic case study, and Chapter 8B (Practical Issues) catalogues the seven pitfalls in detail.